In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: D:\flyrank\.venv\Scripts\python.exe -m pip install --upgrade pip


# ML-02 — Research Question and Provisional Lane

Author: Hamza Afzal · 9 October 2026.

This notebook contains the framing and measured starter evidence. Canonical deliverable: `work/notebooks/w01_research_question.ipynb`. The separate task folder is an index to this notebook; the original assignment structure is preserved.

## 1. My lane and why

**Ranking Signal Analysis**, with explainable content-review priorities as the decision output. I will study which content, freshness, search and engagement signals are associated with observed impression decline across held-out clients. This lane is explicitly supported by the 30,000-row starter dataset in the repository lane guide. It supports useful signal auditing and a transparent queue before a more demanding warehouse time-series study. A classifier is an association tool here; the deliverable is a signal report and careful editorial recommendations.

In [2]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display, Image
candidate = Path.cwd().resolve()
ROOT = next((path for path in [candidate, *candidate.parents]
             if (path / "data/raw/content_refresh_anonymized.csv").exists()), None)
if ROOT is None:
    own_repo = "https://github.com/Hamza2-2/flyrank-ai-internship"
    clone_root = candidate / "flyrank-ai-internship"
    if not clone_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", own_repo, str(clone_root)],
                       check=True, capture_output=True, text=True)
    if not (clone_root / "data/raw/content_refresh_anonymized.csv").exists():
        raise FileNotFoundError("The owned repository checkout has no starter CSV; verify the public repository and rerun.")
    ROOT = clone_root.resolve()
sys.path.insert(0, str(ROOT / "work/scripts"))
from capstone_analysis import load_data, FEATURES, BANNED, run_study
frame, features, target, numbers = load_data()
print('Chosen lane: Ranking Signal Analysis')
print('Unit: one pseudonymized content item; client used only for grouping')

Chosen lane: Ranking Signal Analysis
Unit: one pseudonymized content item; client used only for grouping


## 2. The question: decision, action, cost of a wrong call

**Question:** On clients withheld from model fitting, which safe content and current-window performance signals distinguish pages with measured impression decline, and can these signals prioritize an editor's first 50 reviews better than a transparent freshness/visibility rule?

**Decision and actor:** An SEO/content editor chooses which sufficiently visible pages to inspect first. **Unit:** one page; the client is the independent validation group. **Output:** a signal report and ranked review cases, each with an inspectable reason code. **Action:** verify the latest search evidence, then check metadata, factual freshness, coverage or instrumentation; edits require human judgment. **Cost:** an incorrect priority consumes editor time or risks unnecessary changes; a missed priority delays attention to a potentially important decline. No costs are invented in currency.

**Why data can help:** the transparent rule is the starting point, because it is cheap and explainable. Multiple correlated signals may combine differently across clients, so a fixed logistic model and conservative forest can test whether association beyond that rule is measurable. If they fail, retain the rule. **Metric fixed before fitting:** average precision and precision@50 against the held-out proxy base rate; inspect client variability.

**Target caution:** `trend_direction == 'down'` is a deterministic bucket of an observed current-window change, not an observed future outcome or a label of editorial benefit. Future decline requires earlier feature windows and a later warehouse outcome.

In [3]:
assert not set(features.columns) & BANNED
assert frame.content_id.is_unique
print('Duplicate page IDs:', int(frame.content_id.duplicated().sum()))
print('Forbidden model features:', sorted(set(features.columns) & BANNED))

Duplicate page IDs: 0
Forbidden model features: []


## 3. Quick look at the data: real numbers

The code loads the bundled CSV locally and publishes only aggregates. Row counts establish available evidence; the current decline proxy and stale-visible count establish why a content-review signal study is worth pursuing. The counts are measured, not borrowed from the example pipeline.

In [4]:
support = pd.DataFrame([
    {'measure': 'Starter content items', 'value': numbers['rows']},
    {'measure': 'Client validation groups', 'value': numbers['clients']},
    {'measure': 'Current-decline proxy items', 'value': numbers['declining_rows']},
    {'measure': 'Current-decline proxy share', 'value': round(numbers['declining_share'], 4)},
    {'measure': 'Stale and visible: update >=180 days, impressions >=500', 'value': numbers['stale_visible_rows']},
    {'measure': 'Missing position (zero sentinel)', 'value': numbers['no_position_rows']},
])
display(support)
print('The share is a fraction; source rate columns such as ctr use percentage units.')

,measure,value
0,Starter content items,30000.0000
1,Client validation groups,32.0000
2,Current-decline proxy items,16262.0000
3,Current-decline proxy share,0.5421
4,"Stale and visible: update >=180 days, impressi...",17.0000
5,Missing position (zero sentinel),1205.0000


The share is a fraction; source rate columns such as ctr use percentage units.


## 4. Careful words: what I can and cannot claim

I can report **observed associations**, measured discrimination on unseen client groups, negative results, and **decision-support** review priorities. I cannot claim future decline prediction, Google's ranking factors, causal refresh impact, a guaranteed useful edit, or generalization to the full warehouse. Aggregate 90-day features overlap the proxy window, even after excluding trend fields; that prevents a prospective claim. Position zero is missing measurement, rate columns are percentages, and missing depth/keyword data must be flagged. No IDs, trend fields, provider/model names or six last/previous-30-day trend inputs enter the features.

The full-warehouse extension is pending authorized Hugging Face access. The starter study can run without that account.

In [5]:
print(json.dumps({
    'rates_are_percentages': True,
    'position_zero_is_missing': True,
    'grouping_id_in_features': bool({'client_id','content_id'} & set(features.columns)),
    'future_outcome_available': False,
    'public_outputs_are_aggregate': True,
}, indent=2))

{
  "rates_are_percentages": true,
  "position_zero_is_missing": true,
  "grouping_id_in_features": false,
  "future_outcome_available": false,
  "public_outputs_are_aggregate": true
}


## 5. Self-check

- [x] One supported lane, one decision, one actor and concrete review actions.
- [x] Page unit, output, error cost and a fair rule comparison are stated.
- [x] More than two measured supporting numbers are generated from the CSV.
- [x] ID/label exclusions and same-window limitations are explicit.
- [x] Notebook executed top to bottom after authoring.
- [ ] Intern validates the framing as their own work.
- [ ] Commit to an owned public repository and submit that repository URL.

Assistant assistance: code and document drafting. Hamza must review the decisions and explain the method; this notebook does not claim personal manual work or a portal submission.